# Customer Churn Analysis
> Identifying key drivers of customer churn in a telecom company

**Dataset:** Telco Customer Churn (7,043 rows × 21 columns)  
**Goal:** Find what makes customers leave and build a dashboard to present findings

## Step 1 — Install & Import Libraries

In [ ]:
# All these come pre-installed in Colab — no pip needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Make charts look clean
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('Libraries loaded successfully')

## Step 2 — Load the Dataset

In [ ]:
# Load directly from URL — no download needed
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Basic info — data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

**Observations so far:**
- TotalCharges is loaded as object (string) — needs fixing
- Churn column is Yes/No — needs to be converted to 1/0
- CustomerID is not useful for analysis

## Step 3 — Data Cleaning

In [ ]:
# Check missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Fix TotalCharges: convert to numeric (spaces become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# How many became NaN?
print(f'NaN in TotalCharges after conversion: {df["TotalCharges"].isnull().sum()}')

# These are new customers (tenure=0), fill with 0 makes business sense
df['TotalCharges'].fillna(0, inplace=True)

# Convert Churn to binary 0/1
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop CustomerID — not useful
df.drop('customerID', axis=1, inplace=True)

# Fix SeniorCitizen — already 0/1 but stored as int, convert to readable label for charts
df['SeniorCitizenLabel'] = df['SeniorCitizen'].map({0: 'Non-Senior', 1: 'Senior'})

print('Cleaning complete!')
print(f'Final shape: {df.shape}')
df.head()

In [ ]:
# Save cleaned data (you can download this later)
df.to_csv('churn_clean.csv', index=False)
print('Saved to churn_clean.csv')

## Step 4 — Exploratory Data Analysis (EDA)

### 4.1 — Overall Churn Rate

In [ ]:
churn_rate = df['Churn'].mean() * 100
counts = df['Churn'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Stayed', 'Churned'], counts.values, color=['#1D9E75', '#D85A30'], width=0.5)
axes[0].set_title(f'Customer Churn Count (Churn Rate: {churn_rate:.1f}%)', fontsize=13)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Stayed', 'Churned'],
            autopct='%1.1f%%', colors=['#1D9E75', '#D85A30'],
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Churn Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('chart1_churn_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Insight: {churn_rate:.1f}% of customers churned — about 1 in 4 customers.')
print('This is significant enough for the business to prioritize retention strategies.')

### 4.2 — Churn by Contract Type (Biggest Insight)

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].mean() * 100
contract_counts = df.groupby('Contract')['Churn'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by contract
bars = axes[0].bar(contract_churn.index, contract_churn.values,
                   color=['#D85A30', '#EF9F27', '#1D9E75'], width=0.5)
axes[0].set_title('Churn Rate by Contract Type', fontsize=13)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 55)
for bar, val in zip(bars, contract_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Count by contract
sns.countplot(data=df, x='Contract', hue=df['Churn'].map({1:'Churned', 0:'Stayed'}),
              palette={'Churned': '#D85A30', 'Stayed': '#1D9E75'}, ax=axes[1])
axes[1].set_title('Customer Count by Contract Type', fontsize=13)
axes[1].set_ylabel('Count')
axes[1].legend(title='')

plt.tight_layout()
plt.savefig('chart2_contract_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Month-to-month customers churn at ~42%, compared to just ~3% for two-year contracts.')
print('Recommendation: Offer incentives to migrate month-to-month customers to annual contracts.')

### 4.3 — Churn by Customer Tenure

In [ ]:
# Create tenure buckets
df['TenureBucket'] = pd.cut(df['tenure'],
                             bins=[0, 12, 24, 48, 72],
                             labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
df['ChurnLabel'] = df['Churn'].map({1: 'Churned', 0: 'Stayed'})
sns.boxplot(data=df, x='ChurnLabel', y='tenure',
            palette={'Churned': '#D85A30', 'Stayed': '#1D9E75'}, ax=axes[0])
axes[0].set_title('Tenure Distribution: Churned vs Stayed', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Tenure (months)')

# Churn rate by tenure bucket
tenure_churn = df.groupby('TenureBucket', observed=True)['Churn'].mean() * 100
axes[1].bar(tenure_churn.index, tenure_churn.values,
            color=['#D85A30', '#EF9F27', '#5DCAA5', '#1D9E75'], width=0.5)
axes[1].set_title('Churn Rate by Tenure Group', fontsize=13)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 60)
for i, val in enumerate(tenure_churn.values):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart3_tenure_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: New customers (0-12 months) churn at ~47%, dropping sharply for long-term customers.')
print('Recommendation: Focus retention efforts on customers in their first year.')

### 4.4 — Churn by Monthly Charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE / distribution plot
df[df['Churn']==0]['MonthlyCharges'].plot(kind='kde', ax=axes[0],
                                           label='Stayed', color='#1D9E75', linewidth=2)
df[df['Churn']==1]['MonthlyCharges'].plot(kind='kde', ax=axes[0],
                                           label='Churned', color='#D85A30', linewidth=2)
axes[0].set_title('Monthly Charges Distribution by Churn', fontsize=13)
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].legend()
axes[0].fill_between(axes[0].lines[0].get_xdata(), axes[0].lines[0].get_ydata(), alpha=0.1, color='#1D9E75')
axes[0].fill_between(axes[0].lines[1].get_xdata(), axes[0].lines[1].get_ydata(), alpha=0.1, color='#D85A30')

# Charges bucket churn rate
df['ChargesBucket'] = pd.cut(df['MonthlyCharges'],
                              bins=[0, 35, 65, 95, 120],
                              labels=['Low ($0-35)', 'Mid ($35-65)', 'High ($65-95)', 'Very High ($95+)'])
charges_churn = df.groupby('ChargesBucket', observed=True)['Churn'].mean() * 100
axes[1].bar(charges_churn.index, charges_churn.values,
            color=['#1D9E75', '#EF9F27', '#D85A30', '#993C1D'], width=0.5)
axes[1].set_title('Churn Rate by Monthly Charges', fontsize=13)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 55)
for i, val in enumerate(charges_churn.values):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart4_charges_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Higher monthly charges strongly correlate with churn.')
print('Customers paying $65+ per month churn at over 30% — price sensitivity is a key driver.')

### 4.5 — Churn by Internet Service & Other Services

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Internet service
internet_churn = df.groupby('InternetService')['Churn'].mean() * 100
axes[0].bar(internet_churn.index, internet_churn.values,
            color=['#D85A30', '#EF9F27', '#1D9E75'], width=0.5)
axes[0].set_title('Churn by Internet Service', fontsize=12)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 55)
for i, val in enumerate(internet_churn.values):
    axes[0].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Tech support
tech_churn = df.groupby('TechSupport')['Churn'].mean() * 100
axes[1].bar(tech_churn.index, tech_churn.values,
            color=['#D85A30', '#1D9E75', '#888780'], width=0.5)
axes[1].set_title('Churn by Tech Support', fontsize=12)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 55)
for i, val in enumerate(tech_churn.values):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Payment method
pay_churn = df.groupby('PaymentMethod')['Churn'].mean() * 100
axes[2].barh(pay_churn.index, pay_churn.values,
             color=['#D85A30', '#EF9F27', '#5DCAA5', '#1D9E75'])
axes[2].set_title('Churn by Payment Method', fontsize=12)
axes[2].set_xlabel('Churn Rate (%)')
for i, val in enumerate(pay_churn.values):
    axes[2].text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('chart5_services_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insights:')
print('- Fiber optic customers churn the most despite paying more')
print('- Customers without tech support churn at ~41% vs ~15% with support')
print('- Electronic check payment users churn significantly more than auto-pay users')

### 4.6 — Correlation Heatmap

In [ ]:
# Select only numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['SeniorCitizen']]  # already captured elsewhere

corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, center=0, linewidths=0.5,
            annot_kws={'size': 11})
plt.title('Correlation Matrix — Numeric Features', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('chart6_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: tenure and TotalCharges are highly correlated (expected — longer customers pay more total).')
print('MonthlyCharges has a modest positive correlation with Churn.')

## Step 5 — Summary of Findings

In [ ]:
print('=' * 60)
print('CUSTOMER CHURN ANALYSIS — KEY FINDINGS')
print('=' * 60)

churn_rate = df['Churn'].mean() * 100
mo_churn = df[df['Contract']=='Month-to-month']['Churn'].mean() * 100
two_yr_churn = df[df['Contract']=='Two year']['Churn'].mean() * 100
new_cust_churn = df[df['TenureBucket']=='0-12 months']['Churn'].mean() * 100
no_support_churn = df[df['TechSupport']=='No']['Churn'].mean() * 100
fiber_churn = df[df['InternetService']=='Fiber optic']['Churn'].mean() * 100

print(f'\n Overall churn rate: {churn_rate:.1f}%')
print(f'\n Contract type impact:')
print(f'   Month-to-month: {mo_churn:.1f}% churn')
print(f'   Two-year: {two_yr_churn:.1f}% churn')
print(f'   --> {mo_churn/two_yr_churn:.0f}x higher churn on monthly contracts')
print(f'\n New customers (0-12 months) churn at {new_cust_churn:.1f}%')
print(f'\n Customers without tech support: {no_support_churn:.1f}% churn')
print(f'\n Fiber optic customers: {fiber_churn:.1f}% churn despite higher costs')
print('\n Top 3 Recommendations:')
print('   1. Convert month-to-month customers to annual plans with discounts')
print('   2. Create a 90-day onboarding program for new customers')
print('   3. Bundle tech support into standard plans to reduce churn')

## Next Step — Build the Streamlit Dashboard

Download the chart PNGs and `churn_clean.csv` from the Colab file panel (left sidebar → Files icon).

Then create a new file called `app.py` and paste the Streamlit code to build your interactive dashboard.